# 01 – Cleaning the DBAASP peptide data

**Goal:** turn the raw `by_origin/*.csv` files into one clean table that the later notebooks can use for comparing groups.

**Why `by_origin`?** Every peptide is in exactly one of `Natural.csv` / `Synthetic.csv`, so stacking them rebuilds the full master table. The other folders (`by_kingdom`, `by_length_class`, ...) are just re-slices of the same peptides, so we don't need to load them.

**Cleaning steps** (each one logs how many rows it removes):
1. Load and stack Natural + Synthetic
2. Remove exact duplicate rows
3. Keep monomers only
4. Fix the sequence column (D-amino acids, unknown residues `X`)
5. Remove sequences shorter than 2 residues
6. Turn the multi-label `TARGET GROUP` column into yes/no columns
7. Tidy the kingdom column and run sanity checks
8. Save `results/clean_peptides.csv` and `results/cleaning_log.csv`

In [1]:
import ast
import pandas as pd
import matplotlib.pyplot as plt

DATA = ".."          # repo root, relative to this notebook
OUT = "../results"

# We record (step name, rows left) after every step -> becomes the cleaning log
log = []
def record(step, df):
    origin = df["ORIGIN"] if "ORIGIN" in df else df["origin"]   # column is renamed at the end
    log.append({"step": step, "rows": len(df),
                "natural": int((origin == "Natural").sum()),
                "synthetic": int((origin == "Synthetic").sum())})
    print(f"{step:<45} rows = {len(df):>6}")

## 1. Load and stack Natural + Synthetic

In [2]:
raw = pd.concat(
    [pd.read_csv(f"{DATA}/by_origin/Natural.csv"),
     pd.read_csv(f"{DATA}/by_origin/Synthetic.csv")],
    ignore_index=True,
)
record("1. raw (Natural + Synthetic)", raw)
raw.head(3)

1. raw (Natural + Synthetic)                  rows =  25542


,ID,COMPLEXITY,NAME,N TERMINUS,SEQUENCE,C TERMINUS,SYNTHESIS TYPE,TARGET GROUP,TARGET OBJECT,tokenised_TARGET GROUP,tokenised_TARGET OBJECT,ORIGIN,Length_of_Sequence,Length_Class,KINGDOM,DOMAIN
0,1,Multimer,Distinctin,NaN,NLVSGLIEARKYLEQLHRKLKNCKV ENREVPPGFTALIKTLR...,NaN,Ribosomal,Gram+ Gram-,Lipid Bilayer,"['Gram+', 'Gram-']",['Lipid Bilayer'],Natural,51,Long (50–100 aa),Unclassified,NaN
1,3,Multimer,Halocidin,NaN,WLNALLHHGLNCAKGVLA ALLHHGLNCAKGVLA,AMD AMD,Ribosomal,Gram+,Lipid Bilayer,['Gram+'],['Lipid Bilayer'],Natural,37,Medium (25–50 aa),Unclassified,NaN
2,6,Multi-Peptide,Enterocin X,NaN,SNDSLWYGVGQFMGKQANCITNHPVKHMIIPGYCLSKILG IA...,NaN,Ribosomal,Gram+,Lipid Bilayer,['Gram+'],['Lipid Bilayer'],Natural,81,Long (50–100 aa),Unclassified,NaN


## 2. Remove exact duplicates
The raw stack has more rows than unique IDs. Let's check that the extra rows are exact copies (and not different peptides sharing an ID).

In [3]:
print("rows:", len(raw), "| unique IDs:", raw["ID"].nunique(), "| exact duplicate rows:", raw.duplicated().sum())

df = raw.drop_duplicates().copy()
assert df["ID"].is_unique, "some IDs still repeat with different content"
record("2. drop exact duplicates", df)

rows: 25542 | unique IDs: 23542 | exact duplicate rows: 2000
2. drop exact duplicates                      rows =  23542


## 3. Keep monomers only
`COMPLEXITY` is Monomer, Multimer or Multi-Peptide. Multimers and multi-peptides store several sequences in one cell (separated by spaces), so per-peptide sequence features like charge would be wrong. They are only ~3% of the data.

In [4]:
print(df["COMPLEXITY"].value_counts())
df = df[df["COMPLEXITY"] == "Monomer"].copy()
record("3. keep monomers only", df)

COMPLEXITY
Monomer          22907
Multimer           405
Multi-Peptide      230
Name: count, dtype: int64
3. keep monomers only                         rows =  22907


## 4. Fix the sequence column
Two things in DBAASP sequences need a decision:

* **Lower-case letters** = D-amino acids (mirror-image versions of the normal L-form). We change them to upper case so the amino acid is counted, but keep a flag `has_D_aa` in case we want to compare them later.
* **`X`** = unknown / non-standard residue. We can't count its charge or hydrophobicity, so sequences containing `X` are removed.

In [5]:
df["has_D_aa"] = df["SEQUENCE"].str.contains("[a-z]")
df["SEQUENCE"] = df["SEQUENCE"].str.upper()

standard = df["SEQUENCE"].str.fullmatch("[ACDEFGHIKLMNPQRSTVWY]+")
print("sequences with a non-standard residue:", (~standard).sum())
print("share containing non-standard residues by origin (should be similar, otherwise we bias the comparison):")
print((~standard).groupby(df["ORIGIN"]).mean().round(3))

df = df[standard].copy()
record("4. drop sequences with X / non-standard", df)

sequences with a non-standard residue: 4632
share containing non-standard residues by origin (should be similar, otherwise we bias the comparison):
ORIGIN
Natural      0.188
Synthetic    0.205
Name: SEQUENCE, dtype: float64
4. drop sequences with X / non-standard       rows =  18275


## 5. Remove sequences shorter than 2 residues
DBAASP's own `Length_Class` calls these "Out of range (<2 aa)"; a single residue is not really a peptide.

In [6]:
assert (df["Length_of_Sequence"] == df["SEQUENCE"].str.len()).all(), "length column disagrees with sequence"
df = df[df["Length_of_Sequence"] >= 2].copy()
record("5. drop length < 2", df)

5. drop length < 2                            rows =  18196


## 6. Turn `TARGET GROUP` into yes/no columns
`TARGET GROUP` is multi-label (e.g. `Gram+ Gram- Cancer`). We make one 0/1 column per target so we can later ask "what fraction of natural peptides hit Gram+?".
Rows with no target information stay in, but get `has_target_info = False` so we can leave them out of target comparisons.

In [7]:
df["targets"] = df["tokenised_TARGET GROUP"].apply(ast.literal_eval)

target_cols = {
    "Gram+": "is_gram_pos", "Gram-": "is_gram_neg", "Fungus": "is_fungus",
    "Cancer": "is_cancer", "Virus": "is_virus", "Biofilm": "is_biofilm",
    "Parasite": "is_parasite", "Insect": "is_insect", "Nematode": "is_nematode",
    "Mollicute": "is_mollicute", "Protista": "is_protista", "Archaea": "is_archaea",
    "Mammalian Cell": "is_mammalian_cell",   # = toxic / active on mammalian cells
}
for label, col in target_cols.items():
    df[col] = df["targets"].apply(lambda t: int(label in t))

df["has_target_info"] = df["targets"].str.len() > 0
print("rows without target info:", (~df["has_target_info"]).sum())
df[list(target_cols.values())].sum().sort_values(ascending=False)

rows without target info:

 291


is_gram_neg          14791
is_gram_pos          14178
is_mammalian_cell    10305
is_fungus             5334
is_cancer             3134
is_virus              1252
is_biofilm             613
is_parasite            322
is_insect               96
is_nematode             44
is_protista             44
is_mollicute            34
is_archaea               2
dtype: int64

## 7. Tidy kingdom, pick columns, sanity checks
`KINGDOM` is "Unclassified" for almost all synthetic peptides (they have no natural source). We add a simple `kingdom_known` flag and label the few multi-kingdom rows as "Multiple".

In [8]:
df["KINGDOM"] = df["KINGDOM"].replace({"Bacteria, Fungi": "Multiple"})
df["kingdom_known"] = df["KINGDOM"] != "Unclassified"

keep = (["ID", "NAME", "SEQUENCE", "Length_of_Sequence", "Length_Class", "ORIGIN",
         "SYNTHESIS TYPE", "KINGDOM", "kingdom_known", "has_D_aa", "has_target_info"]
        + list(target_cols.values()))
clean = df[keep].rename(columns={
    "SEQUENCE": "sequence", "Length_of_Sequence": "length", "Length_Class": "length_class",
    "ORIGIN": "origin", "SYNTHESIS TYPE": "synthesis_type", "KINGDOM": "kingdom", "NAME": "name",
}).reset_index(drop=True)

# --- sanity checks: the notebook stops here if something is off ---
assert clean["ID"].is_unique
assert clean["sequence"].str.fullmatch("[ACDEFGHIKLMNPQRSTVWY]+").all()
assert (clean["length"] == clean["sequence"].str.len()).all()
assert clean["length"].min() >= 2
assert set(clean["origin"]) == {"Natural", "Synthetic"}
record("6-7. add target flags, tidy (final)", clean)
clean.head()

6-7. add target flags, tidy (final)           rows =  18196


,ID,name,sequence,length,length_class,origin,synthesis_type,kingdom,kingdom_known,has_D_aa,...,is_cancer,is_virus,is_biofilm,is_parasite,is_insect,is_nematode,is_mollicute,is_protista,is_archaea,is_mammalian_cell
0,11,"Cathelicidin-1, CATH-1, Fowlicidin-1",RVKRVWPLVIRTVIAGYNLYRAIKKK,26,Medium (25–50 aa),Natural,Ribosomal,Animalia,True,False,...,1,0,0,0,0,0,0,0,0,1
1,14,Brevinin-2 related peptide B2RP,GIWDTIKSMGKVFAGKILQNL,21,Short (10–24 aa),Natural,Ribosomal,Animalia,True,False,...,0,1,0,0,0,0,0,0,0,1
2,15,Histatin 5,DSHAKRHHGYKRKFHEKHHSHRGY,24,Short (10–24 aa),Natural,Ribosomal,Animalia,True,False,...,0,0,0,1,0,0,0,0,0,1
3,16,Distinctin chain 1,ENREVPPGFTALIKTLRKCKII,22,Short (10–24 aa),Natural,Ribosomal,Animalia,True,False,...,0,0,0,0,0,0,0,0,0,0
4,36,Distinctin chain 2,NLVSGLIEARKYLEQLHRKLKNCKV,25,Medium (25–50 aa),Natural,Ribosomal,Animalia,True,False,...,0,0,0,0,0,0,0,0,0,0


## Cleaning log

In [9]:
log_df = pd.DataFrame(log)
log_df["removed"] = (log_df["rows"].shift(1) - log_df["rows"]).fillna(0).astype(int)
log_df

,step,rows,natural,synthetic,removed
0,1. raw (Natural + Synthetic),25542,4493,21049,0
1,2. drop exact duplicates,23542,4087,19455,2000
2,3. keep monomers only,22907,3990,18917,635
3,4. drop sequences with X / non-standard,18275,3239,15036,4632
4,5. drop length < 2,18196,3239,14957,79
5,"6-7. add target flags, tidy (final)",18196,3239,14957,0


In [10]:
fig, ax = plt.subplots(figsize=(8, 3.5))
labels = [s.split(". ", 1)[1] for s in log_df["step"]]
ax.barh(labels[::-1], log_df["natural"][::-1], label="Natural")
ax.barh(labels[::-1], log_df["synthetic"][::-1], left=log_df["natural"][::-1], label="Synthetic")
ax.set_xlabel("rows remaining")
ax.set_title("Rows remaining after each cleaning step")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/cleaning_funnel.png", dpi=150)
plt.show()

## Quick look at the final data

In [11]:
print(clean["origin"].value_counts(), "\n")
print(clean.groupby("origin")["length"].describe().round(1), "\n")
print("has_D_aa by origin:\n", clean.groupby("origin")["has_D_aa"].mean().round(3), "\n")
print("has_target_info by origin:\n", clean.groupby("origin")["has_target_info"].mean().round(3))

origin
Synthetic    14957
Natural       3239
Name: count, dtype: int64 

             count  mean   std  min   25%   50%   75%    max
origin                                                      
Natural     3239.0  27.2  16.5  2.0  17.0  25.0  35.0  190.0
Synthetic  14957.0  17.2   9.3  2.0  11.0  15.0  21.0  148.0 

has_D_aa by origin:
 origin
Natural      0.035
Synthetic    0.100
Name: has_D_aa, dtype: float64 

has_target_info by origin:
 origin
Natural      0.992
Synthetic    0.982
Name: has_target_info, dtype: float64


## Save

In [12]:
clean.to_csv(f"{OUT}/clean_peptides.csv", index=False)
log_df.to_csv(f"{OUT}/cleaning_log.csv", index=False)
print("saved", clean.shape, "->", f"{OUT}/clean_peptides.csv")

saved (18196, 24) -> ../results/clean_peptides.csv


## Notes for the methods / limitations section
* ~2,000 exact duplicate rows were removed; IDs are unique afterwards.
* Multimers and multi-peptides were excluded (sequence features are only defined for a single chain).
* D-amino acids were converted to their L-letter (flag kept in `has_D_aa`).
* Sequences with `X` were dropped; check the share dropped per origin (step 4) – if it differs a lot between Natural and Synthetic, mention it as a possible bias.
* The **target-group columns overlap** (a peptide can be Gram+ and Gram- and Cancer). Later comparisons must not treat them as independent groups.
* `kingdom` is known only for natural peptides, so kingdom comparisons apply to Natural only.